# Week 4

## Data Preparation with Pandas / Polars

In [30]:
import pandas as pd
#load as Json from a weblink
url = "https://raw.githubusercontent.com/zhouy185/O712_Datasets/refs/heads/main/grocery_orders.json"
GroceryUrl = pd.read_json(url)

#returns the column headers and a set number of rows.
# GroceryUrl.head()

# returns the data types of the row, aand whether there are any missing values.
GroceryUrl.info()


# #inspecting null values, values indiccated as true are missing.
# GroceryUrl.isnull()

# # We can calculate the sum of of missing values to determine the counts across each column, as a true is equivalent to 1.
GroceryUrl.isnull().sum()




<class 'pandas.DataFrame'>
RangeIndex: 948 entries, 0 to 947
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     948 non-null    int64  
 1   customer_id  948 non-null    str    
 2   store        948 non-null    str    
 3   order_date   948 non-null    int64  
 4   category     948 non-null    str    
 5   quantity     948 non-null    int64  
 6   unit_price   948 non-null    float64
 7   channel      948 non-null    str    
 8   on_time      912 non-null    float64
 9   revenue      948 non-null    float64
 10  complaint    879 non-null    float64
dtypes: float64(4), int64(3), str(4)
memory usage: 109.9 KB


order_id        0
customer_id     0
store           0
order_date      0
category        0
quantity        0
unit_price      0
channel         0
on_time        36
revenue         0
complaint      69
dtype: int64

In [33]:
# Load as xslx file, which has been downloaded.

Grocerydf = pd.read_excel(
    "/home/marty/O712-AnalyticsWPython/Week4/grocery_orders.xlsx",
    sheet_name=0
)

# Grocerydf.head()
Grocerydf.describe()

,order_id,order_date,quantity,unit_price,on_time,revenue,complaint
count,948.000000,948,948.000000,948.000000,912.00000,948.000000,879.000000
mean,20474.500000,2026-06-28 11:08:21.265822,2.919831,5.159873,0.91886,13.729958,0.053470
min,20001.000000,2026-05-25 00:00:00,1.000000,1.250000,0.00000,1.560000,0.000000
25%,20237.750000,2026-06-13 00:00:00,2.000000,3.480000,1.00000,7.227500,0.000000
50%,20474.500000,2026-06-27 12:00:00,3.000000,4.550000,1.00000,12.490000,0.000000
75%,20711.250000,2026-07-14 00:00:00,4.000000,6.225000,1.00000,18.935000,0.000000
max,20948.000000,2026-08-02 00:00:00,8.000000,14.300000,1.00000,39.840000,1.000000
std,273.808327,NaN,1.750519,2.350581,0.27320,7.786014,0.225097


In [ ]:
#Convert's order_date column to date time
# Grocerydf["order_date"] = pd.to_datetime(Grocerydf["order_date"])

Grocerydf.info()
# We can now see that the type has been changed to datetime64. Previously this was a string or integer value.

In [ ]:
# investigating missing values
## Start with filtering for the relevant values.
### we then look at the column we are interested and start working through our assumption.

Grocerydf.query("complaint.isnull()")["order_date"].agg(["max","min"])

max   2026-08-02
min   2026-07-29
Name: order_date, dtype: datetime64[us]

In [62]:
## A common way to address missing columns

# df.fillna({"complaint":False,
#            "on_time":True
#            })

## Alternatively ~ We can drop rows with missing values
# First we would create a copy of the oringal dataframe, such that we don't alter the source
# NewGrocerydf = Grocerydf.dropna(subset=["complaint"])

NewGrocerydf.info()

<class 'pandas.DataFrame'>
RangeIndex: 879 entries, 0 to 878
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     879 non-null    int64         
 1   customer_id  879 non-null    str           
 2   store        879 non-null    str           
 3   order_date   879 non-null    datetime64[us]
 4   category     879 non-null    str           
 5   quantity     879 non-null    int64         
 6   unit_price   879 non-null    float64       
 7   channel      879 non-null    str           
 8   on_time      879 non-null    float64       
 9   revenue      879 non-null    float64       
 10  complaint    879 non-null    bool          
dtypes: bool(1), datetime64[us](1), float64(3), int64(2), str(4)
memory usage: 96.3 KB


In [ ]:
# Converting Complaint from a float or object to a boolean.
NewGrocerydf = NewGrocerydf["complaint"].astype(bool)

In [51]:
NewGrocerydf = Grocerydf.dropna(subset=["complaint"]).copy()
NewGrocerydf["complaint"] = NewGrocerydf["complaint"].astype(bool)

NewGrocerydf["complaint"]

0      False
1      False
2      False
3       True
4      False
       ...  
874    False
875    False
876    False
877    False
878    False
Name: complaint, Length: 879, dtype: bool

In [61]:
print(f"Complaint rate is: {round(NewGrocerydf['complaint'].mean(), 2)}")

Complaint rate is: 0.05


In [64]:
# NewGrocerydf.groupby("store")["complaint"].agg(orders="count","complaints"=sum,"rate"=mean)

NewGrocerydf.groupby("store").agg(
    orders=("complaint", "count"),
    complaints=("complaint", "sum"),
    rate=("complaint", "mean")
)

,orders,complaints,rate
store,,,
Brookhaven,153,5,0.032680
Cedar Falls,220,13,0.059091
Elmsford,295,19,0.064407
Northgate,211,10,0.047393


### Notes
- Workflow for a quick data check
    - Load, Inspect, Display first few rows, perform any necessary conversion

In [ ]:
# Using polars to implement the save activity.

import polars as pl
from io import BytesIO
from urllib.request import urlopen

# Load JSON from a URL
url = "https://raw.githubusercontent.com/zhouy185/O712_Datasets/refs/heads/main/grocery_orders.json"
with urlopen(url) as response:
    GroceryUrl = pl.read_json(BytesIO(response.read()))

# Inspect the data
print(GroceryUrl.schema)
print("Shape:", GroceryUrl.shape)
GroceryUrl.head()


In [ ]:
#using Polars to achieve the same outout as Pandas.

Grocerydf = pl.read_excel(
    "/home/marty/O712-AnalyticsWPython/Week4/grocery_orders.xlsx"
)

print(Grocerydf.schema)
print("Shape:", Grocerydf.shape)
Grocerydf.head()